# Neural Network SMS Text Classifier - Minha Solução

Solução em notebook para o projeto **Neural Network SMS Text Classifier** do freeCodeCamp.

Objetivo: criar uma função `predict_message(pred_text)` que receba uma mensagem SMS e retorne:

```python
[probabilidade, "ham" ou "spam"]
```

- `ham`: mensagem normal
- `spam`: propaganda, golpe, promoção ou mensagem automática


## 1. Importação das bibliotecas

Esta solução usa TensorFlow/Keras com uma rede neural simples para classificação de texto.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)

print("TensorFlow version:", tf.__version__)

## 2. Download dos dados

O projeto usa dois arquivos `.tsv`: um para treino e outro para validação.

In [ ]:
!wget -q https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget -q https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

## 3. Leitura e preparação dos dados

Os rótulos são convertidos assim:

- `ham` → `0`
- `spam` → `1`

In [ ]:
column_names = ["label", "message"]

train_df = pd.read_csv(
    train_file_path,
    sep="\t",
    names=column_names,
    header=None
)

test_df = pd.read_csv(
    test_file_path,
    sep="\t",
    names=column_names,
    header=None
)

label_map = {
    "ham": 0,
    "spam": 1
}

train_df["target"] = train_df["label"].map(label_map).astype("float32")
test_df["target"] = test_df["label"].map(label_map).astype("float32")

train_df.head()

## 4. Separação entre textos e rótulos

Aqui separamos as mensagens (`X`) dos rótulos (`y`).

In [ ]:
train_messages = train_df["message"].astype(str).values
train_labels = train_df["target"].values

test_messages = test_df["message"].astype(str).values
test_labels = test_df["target"].values

print("Mensagens de treino:", len(train_messages))
print("Mensagens de validação:", len(test_messages))
print(train_df["label"].value_counts())

## 5. Vetorização do texto

A camada `TextVectorization` transforma o texto em uma representação numérica.

Nesta solução, uso `tf_idf` com unigramas e bigramas (`ngrams=2`). Isso ajuda o modelo a reconhecer expressões comuns de spam, como chamadas, prêmios, links e promoções.

In [ ]:
max_tokens = 20000

text_vectorizer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="tf_idf",
    ngrams=2,
    standardize="lower_and_strip_punctuation"
)

text_vectorizer.adapt(train_messages)

## 6. Criação da rede neural

O modelo recebe texto como entrada, vetoriza a mensagem e depois usa camadas densas para classificar como `ham` ou `spam`.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(1,), dtype=tf.string),
    text_vectorizer,
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.20),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")]
)

model.summary()

## 7. Treinamento

Como existem muito mais mensagens `ham` do que `spam`, uso `class_weight` para dar mais peso à classe `spam` durante o treinamento.

In [ ]:
class_weight = {
    0: 1.0,
    1: 3.0
}

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    train_messages,
    train_labels,
    epochs=20,
    batch_size=32,
    validation_data=(test_messages, test_labels),
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

## 8. Avaliação do modelo

In [ ]:
loss, accuracy, precision, recall = model.evaluate(test_messages, test_labels, verbose=0)

print(f"Loss: {loss:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

## 9. Escolha automática do melhor limiar

O modelo retorna uma probabilidade entre `0` e `1`. Em vez de usar sempre `0.5`, o notebook procura o melhor limiar na validação usando uma métrica balanceada entre acerto de `ham` e acerto de `spam`.

In [ ]:
validation_probabilities = model.predict(test_messages, verbose=0).flatten()

best_threshold = 0.5
best_score = -1

for threshold in np.arange(0.10, 0.91, 0.01):
    predicted_labels = (validation_probabilities >= threshold).astype(int)

    true_positive = np.sum((predicted_labels == 1) & (test_labels == 1))
    true_negative = np.sum((predicted_labels == 0) & (test_labels == 0))
    false_positive = np.sum((predicted_labels == 1) & (test_labels == 0))
    false_negative = np.sum((predicted_labels == 0) & (test_labels == 1))

    sensitivity = true_positive / (true_positive + false_negative + 1e-8)
    specificity = true_negative / (true_negative + false_positive + 1e-8)

    balanced_accuracy = (sensitivity + specificity) / 2

    if balanced_accuracy > best_score:
        best_score = balanced_accuracy
        best_threshold = threshold

BEST_THRESHOLD = float(best_threshold)

print("Melhor limiar:", BEST_THRESHOLD)
print("Balanced accuracy:", round(best_score, 4))

## 10. Função exigida pelo freeCodeCamp

A função abaixo é a parte principal do projeto. Ela recebe uma mensagem e retorna uma lista no formato:

```python
[probabilidade, "ham" ou "spam"]
```

In [ ]:
def predict_message(pred_text):
    probability = float(model.predict(np.array([pred_text]), verbose=0)[0][0])
    label = "spam" if probability >= BEST_THRESHOLD else "ham"

    return [probability, label]

## 11. Teste final

Esta célula testa a função `predict_message` com as mensagens usadas no desafio.

In [ ]:
def test_predictions():
    test_messages = [
        "how are you doing today",
        "sale today! to stop texts call 98912460324",
        "i dont want to go. can we try it a different day? available sat",
        "our new mobile video service is live. just install on your phone to start watching.",
        "you have won £1000 cash! call to claim your prize.",
        "i'll bring it tomorrow. don't forget the milk.",
        "wow, is your arm alright. that happened to me one time too"
    ]

    test_answers = [
        "ham",
        "spam",
        "ham",
        "spam",
        "spam",
        "ham",
        "ham"
    ]

    passed = True

    for message, answer in zip(test_messages, test_answers):
        prediction = predict_message(message)
        print(prediction, "| Esperado:", answer)

        if prediction[1] != answer:
            passed = False

    if passed:
        print("You passed the challenge. Great job!")
    else:
        print("You haven't passed yet. Keep trying.")

test_predictions()

## 12. Testes extras

Aqui estão alguns exemplos adicionais para verificar o comportamento do classificador.

In [ ]:
extra_messages = [
    "Can you call me when you get home?",
    "Congratulations, you have won a free ticket. Claim now!",
    "Are we still meeting tomorrow morning?",
    "URGENT! Your account has been selected for a cash reward."
]

for message in extra_messages:
    print(message)
    print(predict_message(message))
    print()